# Description

In this notebook, we will explore the merge quantization, from 
- Level 1: per-row/per-column quantization
- Level 2: matrix quantization


In [1]:
import os 
import torch 

In [2]:
def quantization_error_l2_norm(original, dequantized):
    """
    Compute the relative error between the original and dequantized tensors using l2 norm.
    """
    return torch.norm(original - dequantized)


def quantization_error_mse(original, dequantized):
    """
    Compute the Mean Squared Error (MSE) between the original and dequantized tensors.
    """
    return torch.mean((original - dequantized) ** 2)


def quantization_error_kl_divergence(original, dequantized, num_bins=2048, epsilon=1e-10):
    """
    Compute the KL divergence between the distributions of the original and dequantized tensors with float16.
    """
    orig_hist = torch.histc(original.float(), bins=num_bins, min=-1.0, max=1.0)
    deq_hist = torch.histc(dequantized.float(), bins=num_bins, min=-1.0, max=1.0)

    orig_prob = orig_hist / (torch.sum(orig_hist) + epsilon)
    deq_prob = deq_hist / (torch.sum(deq_hist) + epsilon)

    kl_div = torch.sum(orig_prob * torch.log((orig_prob + epsilon) / (deq_prob + epsilon)))
    return kl_div

In [3]:
def quantize_row_matrix_int8_symmetric(mat:torch.Tensor):
    """
    Symmetric quantization to int8 on a per-row basis.
    mat: input float tensor (e.g., torch.float32 or torch.bfloat16)
    """
    N, M = mat.shape
    qmin = -128
    qmax = 127
    
    max_vals, _ = torch.max(torch.abs(mat), dim=1, keepdim=True)  # shape (N, 1)
    scales = (max_vals / qmax).squeeze(1)  # shape (N,)
    
    q_mat = torch.clamp(torch.round(mat / scales.unsqueeze(1)), qmin, qmax).to(torch.int8)  # shape (N, M)
    
    scales = scales.to(torch.float32)
    return q_mat, scales

def de_quantize_row_matrix_int8_symmetric(q_mat:torch.Tensor, scale:torch.Tensor, out_dtype=torch.float16):
    """
    Dequantize int8 matrix to float on a per-row basis.
    q_mat: input int8 tensor (shape (N, M))
    scales: scale factors for each row (shape (N,))
    """
    output = q_mat.to(torch.float32) 
    output = output * scale[:, None]
    output = output.to(out_dtype)
    return output

def quantized_column_matrix_int_symmetric(mat:torch.Tensor):
    """
    Symmetric quantization to int8 on a per-column basis.
    mat: input float tensor (e.g., torch.float32 or torch.bfloat16)
    """
    N, M = mat.shape
    qmin = -128
    qmax = 127
    
    max_vals, _ = torch.max(torch.abs(mat), dim=0, keepdim=True)  # shape (1, M)
    scales = (max_vals / qmax).squeeze(0)  # shape (M,)
    
    q_mat = torch.clamp(torch.round(mat / scales.unsqueeze(0)), qmin, qmax).to(torch.int8)  # shape (N, M)
    
    scales = scales.clone().detach().to(torch.float32)
    return q_mat, scales


def dummy_int8_matmul(A_int8:torch.Tensor, B_int:torch.Tensor, out_dtype=torch.int32):
    """
    This is a dummy int8 matrix multiplication function.
    """
    if A_int8.dtype != torch.int8 or B_int.dtype != torch.int8:
        raise ValueError("Both A and B must be int8 tensors.")
    result_float = torch.matmul(A_int8.float(), B_int.float())
    return result_float.to(out_dtype)


def matmul_de_quantize_symmetric(x_q, w_q, x_scale, w_scale,
                                        out_dtype=torch.float16):
    """
    Dequantize int8 matrix to float on a per-row basis.
    q_mat: input int8 tensor (shape (N, M))
    scales: scale factors for each row (shape (N,))
    """
    output = dummy_int8_matmul(x_q, w_q, out_dtype=torch.float32)
    output = output * x_scale[:, None] * w_scale[None, :]
    output = output.to(out_dtype)
    return output

In [4]:
N = 1024
d_type = torch.float16

W = torch.randn(N, N, device='cuda', dtype=d_type)
X = torch.randn(N, N, device='cuda', dtype=d_type)

A = torch.matmul(X, W)
print(f"Shape of A: {A.shape}, dtype: {A.dtype}")

Shape of A: torch.Size([1024, 1024]), dtype: torch.float16


In [5]:
X_q, x_scale = quantize_row_matrix_int8_symmetric(X)
print(f"Shape of X_q: {X_q.shape}, dtype: {X_q.dtype}")
print(f"Shape of x_scale: {x_scale.shape}, dtype: {x_scale.dtype}")

W_q, w_scale = quantized_column_matrix_int_symmetric(W)
print(f"Shape of W_q: {W_q.shape}, dtype: {W_q.dtype}")
print(f"Shape of w_scale: {w_scale.shape}, dtype: {w_scale.dtype}")

Shape of X_q: torch.Size([1024, 1024]), dtype: torch.int8
Shape of x_scale: torch.Size([1024]), dtype: torch.float32
Shape of W_q: torch.Size([1024, 1024]), dtype: torch.int8
Shape of w_scale: torch.Size([1024]), dtype: torch.float32


In [6]:
A_deq = matmul_de_quantize_symmetric(X_q, W_q, x_scale, w_scale, out_dtype=d_type)
print(f"Shape of A_q: {A_deq.shape}, dtype: {A_deq.dtype}")

Shape of A_q: torch.Size([1024, 1024]), dtype: torch.float16


In [7]:
if torch.allclose(A, A_deq, rtol=2.0, atol=2.0):
    print("Correct !! \n")
else:
    print("WRONG - Dequantized matrix is NOT close !! \n")
    
error_l2 = quantization_error_l2_norm(A, A_deq)
print(f"Quantization L2 norm error (hierarchical quantization int8): {error_l2.item():.6f}")

error_mse = quantization_error_mse(A, A_deq)
print(f"Quantization MSE (hierarchical quantization int8): {error_mse.item():.6f}")

error_kl = quantization_error_kl_divergence(A, A_deq)
print(f"Quantization KL divergence (hierarchical quantization int8): {error_kl.item():.6f}")

Correct !! 

Quantization L2 norm error (hierarchical quantization int8): 364.250000
Quantization MSE (hierarchical quantization int8): 0.126465
Quantization KL divergence (hierarchical quantization int8): 0.081500


What is the value of vector scales

In [8]:
print(f"Value of x_scale: {x_scale}")

print(f"Mean of x_scale: {torch.mean(x_scale).item():.6f}")
print(f"Std of x_scale: {torch.std(x_scale).item():.6f}")

Value of x_scale: tensor([0.0279, 0.0235, 0.0268,  ..., 0.0312, 0.0285, 0.0280], device='cuda:0')
Mean of x_scale: 0.027008
Std of x_scale: 0.002559


In [9]:
print(f"Value of w_scale: {w_scale}")

print(f"Mean of w_scale: {torch.mean(w_scale).item():.6f}")
print(f"Std of w_scale: {torch.std(w_scale).item():.6f}")

Value of w_scale: tensor([0.0273, 0.0222, 0.0287,  ..., 0.0281, 0.0347, 0.0293], device='cuda:0')
Mean of w_scale: 0.027169
Std of w_scale: 0.002645


## COMMENT:
- Both scale of X and scale of W are close. 